# Select Seed Trajectories

For each model, save trajectories with lowest and highest mean cycle amplitude as seed trajectories.

In [1]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
from scipy.signal import find_peaks

# 1. Load trajectories for a given model

In [2]:
AGENT_MODEL = 'value_ppo_unlimited.pth'
# AGENT_MODEL = 'value_ppo_mr_2_1_on_10.pth'
# AGENT_MODEL = 'value_ppo_mr_5_2_on_20.pth'

AGENT_MODEL_NAME = AGENT_MODEL.replace('value_ppo_', '').replace('_on_', '_').replace('.pth', '')

cwd = Path.cwd()
if (cwd / 'initial_trajectories').exists():
    PIPELINE_ROOT = cwd
elif (cwd / 'stimuli' / 'generation_pipeline' / 'initial_trajectories').exists():
    PIPELINE_ROOT = cwd / 'stimuli' / 'generation_pipeline'
else:
    raise FileNotFoundError('Run this notebook from the pipeline folder or the repository root.')

initial_trajectories_root = PIPELINE_ROOT / 'initial_trajectories'
model_trajectory_dir = initial_trajectories_root / AGENT_MODEL_NAME
if model_trajectory_dir.exists():
    trajectory_files = sorted(model_trajectory_dir.glob(f'{AGENT_MODEL_NAME}_trajectory*.json'))
else:
    trajectory_files = sorted(initial_trajectories_root.glob(f'{AGENT_MODEL_NAME}_trajectory*.json'))

trajectory_data = {}
for idx, filepath in enumerate(trajectory_files):
    with open(filepath, 'r') as f:
        data = json.load(f)

    metadata = data.get('metadata', {})
    trajectory = data['trajectory']
    trajectory_data[idx] = {
        'metadata': metadata,
        'trajectory': trajectory,
        'filename': filepath.name,
        'seed': metadata.get('game_seed'),
        'score': metadata.get('final_score'),
        'y_positions': np.array([step['bird']['y'] for step in trajectory]),
        'times': np.array([step.get('time', step_idx) for step_idx, step in enumerate(trajectory)]),
    }

print(f'Loaded {len(trajectory_data)} trajectories from {trajectory_files[0].parent if trajectory_files else initial_trajectories_root}')

Loaded 10 trajectories from /Users/jeanpeic/Cog/enjoyable-action-sequences-cogsci26/stimuli/generation_pipeline/initial_trajectories/unlimited


# 2. Metrics

## 2.1. Jump Cycle Metrics

In [3]:
def compute_jump_cycle_metrics(y_positions, times, prominence=1):
    """Compute simple jump-cycle metrics from the bird y-position trace."""
    y_positions = np.asarray(y_positions)
    times = np.asarray(times)

    peaks, _ = find_peaks(y_positions, prominence=prominence)
    valleys, _ = find_peaks(-y_positions, prominence=prominence)

    cycle_periods = np.diff(times[peaks]) if len(peaks) > 1 else np.array([])

    amplitudes = []
    for peak in peaks:
        adjacent_valleys = []
        previous_valleys = valleys[valleys < peak]
        next_valleys = valleys[valleys > peak]
        if previous_valleys.size:
            adjacent_valleys.append(previous_valleys[-1])
        if next_valleys.size:
            adjacent_valleys.append(next_valleys[0])
        amplitudes.extend(abs(y_positions[peak] - y_positions[valley]) for valley in adjacent_valleys)

    return {
        'cycle_count': len(peaks),
        'mean_cycle_period': float(np.mean(cycle_periods)) if cycle_periods.size else 0.0,
        'std_cycle_period': float(np.std(cycle_periods)) if cycle_periods.size else 0.0,
        'mean_cycle_amplitude': float(np.mean(amplitudes)) if amplitudes else 0.0,
        'max_cycle_amplitude': float(np.max(amplitudes)) if amplitudes else 0.0,
    }

for traj in trajectory_data.values():
    traj['jump_cycle_metrics'] = compute_jump_cycle_metrics(
        traj['y_positions'],
        traj['times'],
    )


## 2.2. Amplitude Range

In [4]:
def compute_amplitude_range(y_positions):
    """Return the total vertical range of a trajectory."""
    y_positions = np.asarray(y_positions)
    return float(np.max(y_positions) - np.min(y_positions))

for traj in trajectory_data.values():
    traj['amplitude_range'] = compute_amplitude_range(traj['y_positions'])

# 3. Summary Table

In [6]:
summary_data = []
for idx, traj in sorted(trajectory_data.items()):
    jump = traj['jump_cycle_metrics']
    summary_data.append({
        'trajectory': idx,
        'filename': traj['filename'],
        'seed': traj['seed'],
        'score': traj['score'],
        'amplitude_range': traj['amplitude_range'],
        'cycle_count': jump['cycle_count'],
        'mean_cycle_period': jump['mean_cycle_period'],
        'std_cycle_period': jump['std_cycle_period'],
        'mean_cycle_amplitude': jump['mean_cycle_amplitude'],
        'max_cycle_amplitude': jump['max_cycle_amplitude'],
    })

df_summary = pd.DataFrame(summary_data).sort_values(by='mean_cycle_amplitude', ascending=False)
df_summary

,trajectory,filename,seed,score,amplitude_range,cycle_count,mean_cycle_period,std_cycle_period,mean_cycle_amplitude,max_cycle_amplitude
4,4,unlimited_trajectory_0005.json,732180,10,268.0,20,0.625514,0.174584,58.052632,196.0
7,7,unlimited_trajectory_0008.json,365838,10,216.0,19,0.640254,0.133348,57.702703,144.0
9,9,unlimited_trajectory_0010.json,131932,10,216.0,19,0.639907,0.142562,57.189189,196.0
3,3,unlimited_trajectory_0004.json,137337,10,228.0,20,0.628247,0.146106,56.184211,225.0
0,0,unlimited_trajectory_0001.json,54886,10,240.0,20,0.624185,0.148753,56.105263,144.0
1,1,unlimited_trajectory_0002.json,110268,10,223.0,20,0.617509,0.150335,54.666667,169.0
6,6,unlimited_trajectory_0007.json,259178,10,219.0,20,0.618649,0.141382,54.256410,169.0
5,5,unlimited_trajectory_0006.json,644167,10,196.0,20,0.618776,0.131831,53.710526,196.0
8,8,unlimited_trajectory_0009.json,121958,10,257.0,20,0.608670,0.146665,53.076923,196.0
2,2,unlimited_trajectory_0003.json,671155,10,198.0,21,0.590581,0.150188,50.575000,196.0


# 4. Saving seed trajectories

In [ ]:
# Copy highest and lowest cycle amplitude trajectories and save them to initial_trajectories folder as AGENT_MODEL_NAME + '_trajectory_high_amplitude.json' and '_trajectory_low_amplitude.json'

import shutil

# Get the highest and lowest cycle amplitude trajectories
high_amplitude_traj = df_summary.iloc[0]
low_amplitude_traj = df_summary.iloc[-1]

# Determine file paths for source trajectories
high_amplitude_src = initial_trajectories_root / AGENT_MODEL_NAME / high_amplitude_traj['filename']
low_amplitude_src = initial_trajectories_root / AGENT_MODEL_NAME / low_amplitude_traj['filename']

# Determine file paths for the destination
high_amplitude_dest = initial_trajectories_root / f'{AGENT_MODEL_NAME}_trajectory_high_amplitude.json'
low_amplitude_dest = initial_trajectories_root / f'{AGENT_MODEL_NAME}_trajectory_low_amplitude.json'

# Copy the files to the new paths
shutil.copyfile(high_amplitude_src, high_amplitude_dest)
shutil.copyfile(low_amplitude_src, low_amplitude_dest)

print(f"Copied high amplitude trajectory: {high_amplitude_src} -> {high_amplitude_dest}")
print(f"Copied low amplitude trajectory: {low_amplitude_src} -> {low_amplitude_dest}")

Copied high amplitude trajectory: /Users/jeanpeic/Cog/enjoyable-action-sequences-cogsci26/stimuli/generation_pipeline/initial_trajectories/unlimited/unlimited_trajectory_0005.json -> /Users/jeanpeic/Cog/enjoyable-action-sequences-cogsci26/stimuli/generation_pipeline/initial_trajectories/2_unlimited_trajectory_high_amplitude.json
Copied low amplitude trajectory: /Users/jeanpeic/Cog/enjoyable-action-sequences-cogsci26/stimuli/generation_pipeline/initial_trajectories/unlimited/unlimited_trajectory_0003.json -> /Users/jeanpeic/Cog/enjoyable-action-sequences-cogsci26/stimuli/generation_pipeline/initial_trajectories/2_unlimited_trajectory_low_amplitude.json
